# Case Study 1: AUTOSAR HLD Document Analysis Assistant

**Project:** Automotive Engineering AI — Standard Project Case Study Document
**Case Study No.:** 1 of 5
**Priority:** Pilot (recommended starting point)

---

## Project Overview
Automate the extraction, organization, search, and validation of architectural knowledge from large **AUTOSAR High-Level Design (HLD)** documents.

## Business Context
HLD documents describe software components, interfaces, ports, signals, dependencies, and functional flows. They are often hundreds of pages long and stored as unstructured PDFs. Architects, developers, integrators, and testers must manually interpret this content, which is slow, inconsistent, and hard to trace.

## Objective
- Reduce manual effort required to review HLD documents
- Create a structured, searchable repository of architecture knowledge
- Improve component, interface, signal, and dependency traceability
- Support design reviews, onboarding, impact analysis, and test planning

## Solution Approach (Reference Architecture)
This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline:

| Stage | Tool used in this notebook |
|---|---|
| Document ingestion | PyMuPDF (`fitz`) |
| Chunking | Section/paragraph-based text splitting |
| Embeddings | Sentence-Transformers (`all-MiniLM-L6-v2`) |
| Vector store | FAISS (local, in-memory) |
| Retrieval + Generation | Top-k similarity search + LLM call (pluggable) |
| Output | Grounded answer with page/section citations |

## Scope
**In scope:** PDF ingestion, component/interface/signal extraction, natural-language Q&A with citations, document comparison.
**Out of scope:** Automatic approval of architecture decisions, modification of source documents, replacement of formal architecture governance.

## Governance Note
All AI-generated output in this notebook is a *review aid only*. Outputs must remain grounded in retrieved source text and are subject to human review before use in any design, safety, or release decision.

---
*Prepared as a working pilot implementation for project evaluation purposes.*


## 1. Environment Setup
Install and import the required libraries.

In [5]:
# Install dependencies (run once)
!pip install pymupdf sentence-transformers faiss-cpu numpy --quiet



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip install pymupdf sentence-transformers faiss-cpu numpy --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
!pip install -U jupyter ipywidgets --quiet



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import fitz  # PyMuPDF
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import textwrap

print("Libraries loaded successfully.")


Libraries loaded successfully.


## 2. Document Ingestion
Upload an AUTOSAR HLD PDF (or any architecture document) and extract text per page.
Replace `PDF_PATH` with the path to your document.

In [9]:
PDF_PATH = r"C:\Users\Ananaya Keerthi\Downloads\AUTOSAR_CP_EXP_LayeredSoftwareArchitecture.pdf"

def extract_pages(pdf_path):
    """Extract text from each page of the PDF, keeping page numbers for citation."""
    doc = fitz.open(pdf_path)
    pages = []

    for i, page in enumerate(doc):
        text = page.get_text("text").strip()
        if text:
            pages.append({"page": i + 1, "text": text})

    doc.close()
    return pages

pages = extract_pages(PDF_PATH)
print(f"Extracted {len(pages)} pages.")

Extracted 185 pages.


## 3. Chunking
Split page text into smaller, context-aware chunks for embedding.

In [10]:
def chunk_pages(pages, chunk_size=800, overlap=100):
    """Split page text into overlapping chunks; retain page number for each chunk."""
    chunks = []

    for p in pages:
        text = p["text"]
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            chunks.append({
                "page": p["page"],
                "text": chunk_text
            })

            start += chunk_size - overlap

    return chunks


chunks = chunk_pages(pages)
print(f"Created {len(chunks)} chunks.")

Created 339 chunks.


## 4. Embeddings and Local Vector Store (FAISS)
Convert chunks into embeddings and index them for semantic search.

In [24]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def build_index(chunks):
    texts = [c["text"] for c in chunks]

    embeddings = embedding_model.encode(
        texts,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    dim = embeddings.shape[1]

    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    return index, embeddings


# Build the FAISS index
index, embeddings = build_index(chunks)

print(f"Indexed {index.ntotal} chunk embeddings.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Indexed 339 chunk embeddings.


## 5. Retrieval
Given a user query, retrieve the top-k most relevant chunks.

In [26]:
def retrieve(query, chunks, index, top_k=4):
    query_vec = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    results = [chunks[i] for i in indices[0]]
    return results


results = retrieve(
    "What interfaces does the ECU communication module expose?",
    chunks,
    index
)

for r in results:
    print(f"[Page {r['page']}]")
    print(textwrap.shorten(r['text'], 150))
    print()


[Page 16]
5acXjzUk R24-11 Document ID 53: AUTOSAR_CP_EXP_LayeredSoftwareArchitecture 26 Architecture – Overview of Software Layers ECU Abstraction Layer [...]

[Page 19]
ach ECU) Upper Interface: completely ECU independent Microcontroller Microcontroller Abstraction Layer Application Layer AUTOSAR Runtime [...]

[Page 33]
5acXjzUk R24-11 Document ID 53: AUTOSAR_CP_EXP_LayeredSoftwareArchitecture 43 Architecture – Content of Software Layers ECU Abstraction: [...]

[Page 23]
5acXjzUk R24-11 Document ID 53: AUTOSAR_CP_EXP_LayeredSoftwareArchitecture 33 Architecture – Introduction to Basic Software Module Types [...]



## 6. Grounded Answer Generation
Combine the retrieved chunks into a prompt and call an LLM to generate a cited answer.
Plug in your approved/local LLM API call inside `call_llm()`.

In [32]:
def build_prompt(query, results):
    context = "\n\n".join(
        [f"[Page {r['page']}]\n{r['text']}" for r in results]
    )

    prompt = f"""You are an AUTOSAR HLD analysis assistant. Answer the question using ONLY the
context below. Cite the page number(s) for every claim. If the answer is not
in the context, say so explicitly.

Context:
{context}

Question: {query}

Answer (with page citations):"""

    return prompt


def call_llm(prompt):
    """
    Plug in your approved LLM here (local model, enterprise API, etc.).
    This is a placeholder that just echoes the prompt length for now.
    """

    # Example placeholder — replace with a real call, e.g.:
    # response = client.messages.create(
    #     model="...",
    #     messages=[{"role": "user", "content": prompt}]
    # )
    # return response.content[0].text

    return "[LLM output placeholder — connect your model in call_llm()]"


def ask(query, chunks, index, top_k=4):
    results = retrieve(query, chunks, index, top_k=top_k)
    prompt = build_prompt(query, results)
    answer = call_llm(prompt)

    print("Q:", query)
    print("\nA:", answer)
    print("\nSources:", [r["page"] for r in results])

    return answer, results


# Test
# ask(
#     "What are the key interfaces and dependencies described in this document?",
#     chunks,
#     index
# )

## 7. End-to-End Pipeline Run
Uncomment and run once `PDF_PATH` is set to a real document.

In [33]:
# --- Full pipeline ---
# pages = extract_pages(PDF_PATH)
# chunks = chunk_pages(pages)
# index, embeddings = build_index(chunks)
# ask("Summarize the main software components described in this document.", chunks, index)
# --- Full pipeline ---

pages = extract_pages(PDF_PATH)
chunks = chunk_pages(pages)
index, embeddings = build_index(chunks)

ask(
    "Summarize the main software components described in this document.",
    chunks,
    index
)


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 11/11 [00:11<00:00,  1.01s/it]


Q: Summarize the main software components described in this document.

A: [LLM output placeholder — connect your model in call_llm()]

Sources: [79, 20, 10, 18]


('[LLM output placeholder — connect your model in call_llm()]',
 [{'page': 79,
   'text': '5acXjzUk\nR24-11\nDocument ID 53: \nAUTOSAR_CP_EXP_LayeredSoftwareArchitecture \n89\nInterfaces\nComponents and interfaces view (simplified)\nECU-Hardware\nAUTOSAR Runtime Environment (RTE)\nActuator\nSoftware\nComponent\nAUTOSAR\nInterface\nApplication\nSoftware\nComponent\nSensor\nSoftware\nComponent\nApplication\nSoftware\nComponent\n..............\nAUTOSAR\nSoftware\nBasic Software\nStandardized\nInterface\nAUTOSAR\nInterface\nAUTOSAR\nInterface\nAUTOSAR\nInterface\nMicrocontroller\nAbstraction\nAUTOSAR\nSoftware\nComponent\nStandard\nSoftware\nStandardized\nAUTOSAR\nInterface\nServices\nStandardized\nInterface\nECU\nAbstraction\nAUTOSAR\nInterface\nStandardized\nInterface\nComplex\nDrivers\nAUTOSAR\nInterface\nVFB & RTE \nrelevant\nStandardized\nInterface\nCommunication\nStandardized\nInterface\nStandardized\nInterface\nOperating\nSystem\nRTE\nrelevant\n'},
  {'page': 20,
   'text': '5acXjzU

In [34]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print(response.json())

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000165A12E6900>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [35]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print(response.json())

{'models': [{'name': 'llama3.2:latest', 'model': 'llama3.2:latest', 'modified_at': '2026-09-09T22:09:08.6695983+05:30', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 3072}, 'capabilities': ['completion', 'tools']}]}


In [15]:
import requests

def call_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:latest",
            "prompt": prompt,
            "stream": False
        }
    )

    response.raise_for_status()

    return response.json()["response"]

print("Ollama LLM connected successfully!")

Ollama LLM connected successfully!


In [37]:
answer, results = ask(
    "What are the main software components described in this document?",
    chunks,
    index
)

Q: What are the main software components described in this document?

A: According to the provided context, the main software components described in this document are:

1. Basic Software (Page 20, [Table of Services])
2. Services Layer (Page 18, [Architecture – Overview of Software Layers])
3. Runtime Environment (RTE) (Page 79, [Interfaces])
4. Application Software Cluster (Page 110, [Application Layer])
5. Software Cluster Connection (SwCluC) (Page 110, [Application Layer])
6. RTE-Plugin (Page 110, [Application Layer])
7. Microcontroller (Abstraction) (Page 79, [Interfaces])
8. Actuator (Page 79, [Interfaces])
9. Sensor (Page 79, [Interfaces])
10. Vehicle-to-X communication (Page 20, [Table of Services])
11. Vehicle network systems (Page 20, [Table of Services])
12. ECU onboard communication systems (Page 20, [Table of Services])
13. ECU internal SW (Page 20, [Table of Services])
14. Vehicle onboard peripherals (Page 20, [Table of Services])
15. ECU onboard peripherals (Page 20, [Ta

In [41]:
def build_prompt(query, results):
    context = "\n\n".join(
        [f"[Page {r['page']}]\n{r['text']}" for r in results]
    )

    prompt = f"""You are an AUTOSAR HLD Document Analysis Assistant.

Answer the user's question using ONLY the provided document context.

Important rules:
1. Do not add information from your general knowledge.
2. Do not invent components, interfaces, or relationships.
3. Distinguish between:
   - Software components
   - Software layers
   - Interfaces
   - Services/functions
4. If the question refers to a diagram, view, table, or architecture section,
report the entities and labels explicitly shown in that source section.
Preserve the terminology used in the document.
Do not reinterpret or rename them.
5. Cite the page number immediately after each relevant statement.
6. If the context does not contain enough information, say:
   "The provided context does not contain enough information to answer this."

Document context:
{context}

Question:
{query}

Answer:"""

    return prompt

In [42]:
answer, results = ask(
    "What entities and interfaces are shown in the components and interfaces view?",
    chunks,
    index,
    top_k=2
)

Q: What entities and interfaces are shown in the components and interfaces view?

A: The entities and interfaces shown in the "Components and Interfaces" view are:

1. Interfaces:
 * Communication Interface
 * Operating System Interface
 * Basic Software Interface
 * System Services Interface
 * Time Service Interface
 * Synchronized Time-base Manager Interface
2. Entities:
 * RTE (Real-Time Engine)
 * BSW (Basic Software Mode Manager)
 * ECU State Manager (EcuM)
 * Watchdog Manager (WdgM)
 * Diagnostic Event Manager (Dem)
 * Function Inhibition Manager (FiM)
 * Microcontroller (µC)

Sources: [79, 61]


In [31]:
def extract_architecture(query, chunks, index, top_k=4):
    results = retrieve(query, chunks, index, top_k=top_k)

    context = "\n\n".join(
        [f"[Page {r['page']}]\n{r['text']}" for r in results]
    )

    prompt = f"""
You are an AUTOSAR HLD architecture analysis assistant.

Use ONLY the document context below.

Extract the architecture information explicitly present in the
provided context.

Return exactly these sections:

COMPONENTS:
- component name — page number

INTERFACES:
- interface name — page number

LAYERS:
- layer name — page number

OTHER ARCHITECTURE ENTITIES:
- entity name — page number

RULES:
- Do not invent information.
- Preserve the terminology used in the document.
- Do not treat a service/function as a component unless the document
  explicitly presents it as one.
- If a category is not present in the context, write "None found".
- Use the page number from the supplied context.

DOCUMENT CONTEXT:
{context}

QUESTION:
{query}

ARCHITECTURE EXTRACTION:
"""

    answer = call_llm(prompt)

    print(answer)

    return answer, results

In [32]:
architecture, architecture_sources = extract_architecture(
    "Extract the components, interfaces, layers and other architecture entities from the document.",
    chunks,
    index,
    top_k=4
)

Here are the extracted architecture information:

COMPONENTS:
- AUTOSAR_CP_EXP_LayeredSoftwareArchitecture — page 33
- Application Layer — page 122
- Drivers (Microcontroller Abstraction Layer) — page 94ju8
- ECU Abstraction Layer — page 18 and page 122
- Microcontroller — page 18
- RTE — page 18
- Services Layer — page 122

INTERFACES:
- CAN communication system interface — page 111
- General interface — page 111

LAYERS:
- ECU Abstraction Layer — page 18
- Application Layer — page 122
- Microcontroller Abstraction Layer — page 18
- Drivers Layer — page 122
- Services Layer — page 122

OTHER ARCHITECTURE ENTITIES:
- None found


In [47]:
def extract_architecture(query, chunks, index, top_k=4):

    # 1. Get normal semantic retrieval
    results = retrieve(query, chunks, index, top_k=top_k)

    # 2. Also force retrieval of important architecture page(s)
    architecture_pages = [79]

    for page_num in architecture_pages:
        for c in chunks:
            if c["page"] == page_num:
                if c not in results:
                    results.append(c)

    # 3. Build context
    context = "\n\n".join(
        [f"[Page {r['page']}]\n{r['text']}" for r in results]
    )

    # 4. Ask Llama to extract only explicitly supported information
    prompt = f"""
You are an AUTOSAR architecture document analysis assistant.

Use ONLY the supplied document context.

Extract architecture information explicitly present in the context.

Return EXACTLY this format:

COMPONENTS:
- Name — Page X

INTERFACES:
- Name — Page X

LAYERS:
- Name — Page X

OTHER ENTITIES:
- Name — Page X

RULES:
1. Do not use general AUTOSAR knowledge.
2. Do not invent or infer entities.
3. Do not treat section headings as components or interfaces.
4. Only include an item when the context explicitly supports it.
5. Keep the terminology used in the document.
6. A service/function should NOT be classified as a component.
7. A layer should NOT be classified as a component.
8. If nothing suitable is found for a category, write "None found".
9. Use the page number provided in the context.

DOCUMENT CONTEXT:
{context}

QUESTION:
{query}

ARCHITECTURE EXTRACTION:
"""

    answer = call_llm(prompt)

    print(answer)

    return answer, results

In [48]:
architecture, architecture_sources = extract_architecture(
    "Identify the architecture components, interfaces, layers and other entities explicitly shown or described in the document.",
    chunks,
    index,
    top_k=4
)

COMPONENTS:
- Name — None found

INTERFACES:
- Name — Page 79
  - Communication
  - Operating System
  - RTE (relevant)
  - BSW (relevant)
  - Basic Software (not specified within AUTOSAR)
- Name — Page 86
  - Interaction of Layers (e.g., "Memory")
- Name — Page 122
  - General
  - Interaction of Layers (Examples)
- Name — Page 79
  - ECU-Hardware
  - Actuator
  - Software
  - Component
  - Sensor
  - Software
  - Component
  - Microcontroller
  - Abstraction
  - Services
  - ECU
  - Abstraction
  - Complex
  - Drivers
  - VFB & RTE
  - Communication
  - Standardized Interface

LAYERS:
- Name — Page 19
  - AUTOSAR Runtime Environment (RTE)
- Name — Page 18
  - Services Layer
- Name — Page 28
  - ECU Abstraction Layer
- Name — Page 28
  - Basic Software Layer

OTHER ENTITIES:
- Name — Page 96
  - Memory service modules


In [35]:
def extract_dependencies(query, chunks, index, top_k=6):

    # Retrieve relevant document sections
    results = retrieve(query, chunks, index, top_k=top_k)

    # Build context with page numbers
    context = "\n\n".join(
        [f"[Page {r['page']}]\n{r['text']}" for r in results]
    )

    prompt = f"""
You are an AUTOSAR HLD document analysis assistant.

Use ONLY the supplied document context.

Identify dependencies or relationships that are EXPLICITLY
described in the document.

Return the answer in exactly this format:

DEPENDENCIES:

1. SOURCE:
   - Name: <source entity>
   - Depends on / interacts with: <target entity>
   - Relationship: <brief description>
   - Page: <page number>

2. SOURCE:
   - Name: <source entity>
   - Depends on / interacts with: <target entity>
   - Relationship: <brief description>
   - Page: <page number>

RULES:
- Use only information explicitly supported by the context.
- Do not use general AUTOSAR knowledge.
- Do not invent relationships.
- Do not assume that two entities are connected just because
  they appear on the same page.
- Preserve terminology from the document.
- Include the page number supporting each relationship.
- If no explicit dependency is found, write:
  "No explicit dependencies found in the provided context."

DOCUMENT CONTEXT:
{context}

QUESTION:
{query}

DEPENDENCY ANALYSIS:
"""

    answer = call_llm(prompt)

    print(answer)

    return answer, results

In [36]:
dependencies, dependency_sources = extract_dependencies(
    "Identify the explicit dependencies and interactions between software components, interfaces, RTE, BSW, and other architecture entities.",
    chunks,
    index,
    top_k=6
)

DEPENDENCIES:

1. SOURCE:
   - Name: RTE
   - Depends on / interacts with: AUTOSAR Software Components
   - Relationship: Provides communication services to application software
   - Page: 29

2. SOURCE:
   - Name: AUTOSAR Software Components
   - Depends on / interacts with: RTE
   - Relationship: Communicate with other components (inter and/or intra ECU) and/or services via the RTE
   - Page: 29

3. SOURCE:
   - Name: RTE
   - Depends on / interacts with: ECU
   - Relationship: Provides communication services to ECU
   - Page: 29

4. SOURCE:
   - Name: ECU
   - Depends on / interacts with: RTE
   - Relationship: Uses the RTE for communication
   - Page: 29

5. SOURCE:
   - Name: Complex Drivers
   - Depends on / interacts with: RTE
   - Relationship: Integrates special purpose functionality, e.g. drivers for devices, with the RTE
   - Page: 27

6. SOURCE:
   - Name: RTE
   - Depends on / interacts with: Complex Drivers
   - Relationship: Enables the integration of special purpose fun

In [39]:
def generate_component_report(component_name, chunks, index, top_k=6):

    query = f"""
    Find all information in the document related to the component:
    {component_name}

    Include its description, interfaces, layers, interactions,
    dependencies, and other explicitly mentioned information.
    """

    results = retrieve(query, chunks, index, top_k=top_k)

    context = "\n\n".join(
        [f"[Page {r['page']}]\n{r['text']}" for r in results]
    )

    prompt = f"""
You are an AUTOSAR HLD Document Analysis Assistant.

Generate a structured report for the requested architecture entity.

Use ONLY the supplied document context.
Do not use general AUTOSAR knowledge.
Do not invent missing information.

Return exactly this format:

COMPONENT REPORT
================

COMPONENT:
<name>

DESCRIPTION:
<information explicitly stated in the document>

INTERFACES:
- <interface> — Page X

LAYERS:
- <layer> — Page X

INTERACTIONS:
- <entity> — <relationship> — Page X

DEPENDENCIES:
- <entity> — <dependency/relationship> — Page X

OTHER INFORMATION:
- <relevant information> — Page X

DOCUMENT REFERENCES:
- Page X
- Page X

RULES:
1. Include only information supported by the supplied context.
2. Every factual statement must have a page citation.
3. Do not invent interfaces, dependencies, or relationships.
4. If a section has no supported information, write "None found".
5. Keep the terminology from the document.
6. Do not confuse a service or function with a component.
7. Do not treat section headings as architecture entities.

DOCUMENT CONTEXT:
{context}

REQUESTED COMPONENT:
{component_name}

COMPONENT REPORT:
"""

    answer = call_llm(prompt)

    print(answer)

    return answer, results

In [40]:
rte_report, rte_sources = generate_component_report(
    "RTE",
    chunks,
    index,
    top_k=6
)


COMPONENT REPORT

COMPONENT:
RTE

DESCRIPTION:
The AUTOSAR Runtime Environment (RTE) is a layer providing communication services to the application software (AUTOSAR Software Components and/or AUTOSAR Sensor/Actuator components). Above the RTE, the software architecture style changes from "layered" to "component style".

INTERFACES:
- SomeIpXf_SOMEIP_Signal1 — Page 103
- SafetyXf_Safety_Signal1 — Page 103

LAYERS:
- Application Layer — Page 83
- System Service Layer — Page 103
- Services Layer — Page 18

INTERACTIONS:
- RTE — E2EWRP — Page 83
- E2EWRP — E2E_P0xCheck() — Page 83

DEPENDENCIES:
- ECU Abstraction Layer — Page 103
- Microcontroller Abstraction Layer — Page 103

OTHER INFORMATION:
- The RTE transfers the final output data as a byte array to the COM module. — Page 102
- The RTE uses a buffer to protect the data and writes the output into another buffer. A new buffer is required because in-place buffer handling is not used. — Page 102

DOCUMENT REFERENCES:
- Page 83
- Page 10

In [22]:
def check_completeness(query, chunks, index, top_k=8):

    # Retrieve relevant sections
    results = retrieve(query, chunks, index, top_k=top_k)

    # Build context with page citations
    context = "\n\n".join(
        [f"[Page {r['page']}]\n{r['text']}" for r in results]
    )

    prompt = f"""
You are an AUTOSAR HLD Document Analysis Assistant.

Analyze ONLY the supplied document context.

Your task is to identify:
1. Missing information
2. Potential inconsistencies or mismatches
3. Areas where the document does not provide enough information

Do NOT use general AUTOSAR knowledge.
Do NOT assume something is missing simply because it is not
present in the retrieved context.

Return exactly this format:

COMPLETENESS / MISMATCH CHECK
=============================

MISSING INFORMATION:
- Finding — Page X
- Finding — Page X

POTENTIAL MISMATCHES:
- Finding — Page X
- Finding — Page X

INCOMPLETE / UNCLEAR AREAS:
- Finding — Page X

WELL-SPECIFIED AREAS:
- Finding — Page X

OVERALL ASSESSMENT:
<brief assessment>

RULES:
- Only report findings supported by the supplied context.
- Do not invent mismatches.
- Do not treat absence from the retrieved context as proof that
  something is absent from the entire document.
- Clearly distinguish "not found in the provided context" from
  "missing from the document".
- Every factual finding must include its page number.
- If no potential mismatch is found, write "None identified".
- If no clear missing information is found, write "None identified".
- Keep terminology from the document.

DOCUMENT CONTEXT:
{context}

QUESTION:
{query}

COMPLETENESS / MISMATCH ANALYSIS:
"""

    answer = call_llm(prompt)

    print(answer)

    return answer, results

In [27]:
completeness, completeness_sources = check_completeness(
    "Check the AUTOSAR architecture information for missing, incomplete, or potentially inconsistent information.",
    chunks,
    index,
    top_k=8
)

MISSING INFORMATION:
- Finding — Page 1 (Document Title and Document Owner information is not explicitly defined, but is mentioned as part of the document header)

POTENTIAL MISMATCHES:
- Finding — Page 11 (AUTOSAR extensibility states that non-standard modules cannot be added to the AUTOSAR Software Architecture, but the term "Complex Drivers" is used without clear definition)

INCOMPLETE / UNCLEAR AREAS:
- Finding — Page 152 (Integration and Runtime Aspects - Error Handling, Reporting and Diagnostic Mechanism in relation to AUTOSAR layers and system life time does not provide clear information on the relationship between Error Reporting and Error Handling)

WELL-SPECIFIED AREAS:
- Finding — Page 163 (Integration and Runtime Aspects - Error Handling, Reporting and Diagnostic Mechanism in relation to AUTOSAR layers and system life time provides clear information on the different error reporting mechanisms)

OVERALL ASSESSMENT:
The document appears to be comprehensive, covering various 

## 8. Revision Comparison

Compares two versions of an HLD document (e.g. an older AUTOSAR release vs a newer one)
and reports **ADDED**, **REMOVED**, **CHANGED**, and **POTENTIAL INCONSISTENCIES**, each
with page references.

**Approach (two layers, matching the reference architecture's own principle that
"deterministic rules are applied where protocol, safety, risk, or validation logic
requires them"):**

1. **Deterministic page-level diff** — `difflib` compares Version A and Version B
   page by page. This step never hallucinates; it either finds text differences or
   it doesn't.
2. **LLM summarization** — Llama 3.2 is given *only* the diff output (not the raw
   documents) and asked to summarize it into a structured report with page citations.
   It cannot invent an addition/removal/change that the diff didn't already find.

This reuses your existing `extract_pages()`, `chunk_pages()`, and `call_llm()` —
no new ingestion or LLM logic is introduced.

In [1]:
import difflib

def extract_and_chunk(pdf_path, chunk_size=800, overlap=100):
    """Reuses the existing extract_pages() + chunk_pages() pipeline for one document."""
    pages = extract_pages(pdf_path)
    chunks = chunk_pages(pages, chunk_size=chunk_size, overlap=overlap)
    return pages, chunks


def diff_pages(pages_a, pages_b, similarity_threshold=0.85):
    """
    Deterministic page-level comparison between two document versions.
    Matches pages by page number and classifies each as ADDED, REMOVED,
    UNCHANGED, or CHANGED based on text similarity (difflib SequenceMatcher).
    No LLM is involved in this step.
    """
    pages_a_map = {p["page"]: p["text"] for p in pages_a}
    pages_b_map = {p["page"]: p["text"] for p in pages_b}

    all_page_numbers = sorted(set(pages_a_map.keys()) | set(pages_b_map.keys()))

    diff_results = []

    for page_num in all_page_numbers:
        text_a = pages_a_map.get(page_num)
        text_b = pages_b_map.get(page_num)

        if text_a is None and text_b is not None:
            diff_results.append({"page": page_num, "status": "ADDED", "text_b": text_b})
        elif text_a is not None and text_b is None:
            diff_results.append({"page": page_num, "status": "REMOVED", "text_a": text_a})
        else:
            ratio = difflib.SequenceMatcher(None, text_a, text_b).ratio()
            if ratio >= similarity_threshold:
                diff_results.append({
                    "page": page_num, "status": "UNCHANGED", "similarity": round(ratio, 3)
                })
            else:
                diff_results.append({
                    "page": page_num,
                    "status": "CHANGED",
                    "similarity": round(ratio, 3),
                    "text_a": text_a,
                    "text_b": text_b
                })

    return diff_results


print("Diff engine ready.")

Diff engine ready.


In [2]:
def summarize_diff_for_llm(diff_results, max_changed_pages=8, snippet_chars=500):
    """
    Build a compact context string for the LLM: every ADDED and REMOVED page,
    plus the most significantly CHANGED pages (lowest similarity first, capped
    at max_changed_pages to keep the prompt within a reasonable size).
    """
    added = [d for d in diff_results if d["status"] == "ADDED"]
    removed = [d for d in diff_results if d["status"] == "REMOVED"]
    changed = sorted(
        [d for d in diff_results if d["status"] == "CHANGED"],
        key=lambda d: d["similarity"]
    )[:max_changed_pages]

    context_parts = []

    for d in added:
        context_parts.append(f"[ADDED - Page {d['page']} in Version B]\n{d['text_b'][:snippet_chars]}")

    for d in removed:
        context_parts.append(f"[REMOVED - Page {d['page']} from Version A]\n{d['text_a'][:snippet_chars]}")

    for d in changed:
        context_parts.append(
            f"[CHANGED - Page {d['page']} - similarity {d['similarity']}]\n"
            f"VERSION A:\n{d['text_a'][:snippet_chars]}\n\n"
            f"VERSION B:\n{d['text_b'][:snippet_chars]}"
        )

    return "\n\n".join(context_parts)


print("Context builder ready.")

Context builder ready.


In [3]:
def compare_revisions(pdf_path_a, pdf_path_b, chunk_size=800, overlap=100,
                       similarity_threshold=0.85, max_changed_pages=8):
    """
    Compare two HLD document revisions.
    Version A = older/baseline document, Version B = newer/updated document.

    Step 1: Deterministic page-level diff (difflib) - no hallucination risk.
    Step 2: LLM (Llama 3.2 via Ollama) summarizes the diff into a structured
            ADDED / REMOVED / CHANGED / POTENTIAL INCONSISTENCIES report,
            grounded only in the diff data - not the raw documents.
    """
    print("Extracting and chunking Version A...")
    pages_a, chunks_a = extract_and_chunk(pdf_path_a, chunk_size, overlap)
    print(f"Version A: {len(pages_a)} pages, {len(chunks_a)} chunks")

    print("Extracting and chunking Version B...")
    pages_b, chunks_b = extract_and_chunk(pdf_path_b, chunk_size, overlap)
    print(f"Version B: {len(pages_b)} pages, {len(chunks_b)} chunks")

    print("Computing deterministic page-level diff...")
    diff_results = diff_pages(pages_a, pages_b, similarity_threshold)

    added_count = sum(1 for d in diff_results if d["status"] == "ADDED")
    removed_count = sum(1 for d in diff_results if d["status"] == "REMOVED")
    changed_count = sum(1 for d in diff_results if d["status"] == "CHANGED")
    unchanged_count = sum(1 for d in diff_results if d["status"] == "UNCHANGED")

    print(f"Diff summary: {added_count} added, {removed_count} removed, "
          f"{changed_count} changed, {unchanged_count} unchanged pages\n")

    context = summarize_diff_for_llm(diff_results, max_changed_pages)

    prompt = f"""
You are an AUTOSAR HLD Document Analysis Assistant performing a revision comparison.

Use ONLY the page-level diff data below, which was computed deterministically
by comparing Version A (older) and Version B (newer) of the same document.

Do NOT use general AUTOSAR knowledge.
Do NOT invent additions, removals, or changes that are not present in the diff data.

Return exactly this format:

REVISION COMPARISON REPORT
===========================

ADDED:
- Finding — Page X (Version B)

REMOVED:
- Finding — Page X (Version A)

CHANGED:
- Finding — Page X (summarize what changed between Version A and Version B)

POTENTIAL INCONSISTENCIES:
- Finding — Page X (e.g. a component/interface described differently, or
  contradicted, between the two versions)

OVERALL ASSESSMENT:
<brief assessment of the scope and nature of the revision>

RULES:
- Only report findings supported by the supplied diff data.
- Every finding must include its page number.
- If a category has no findings, write "None identified".
- Keep terminology from the document.

DIFF DATA:
{context}

REVISION COMPARISON ANALYSIS:
"""

    answer = call_llm(prompt)
    print(answer)

    return {
        "answer": answer,
        "diff_results": diff_results,
        "summary_counts": {
            "added": added_count,
            "removed": removed_count,
            "changed": changed_count,
            "unchanged": unchanged_count,
        }
    }


print("compare_revisions() ready.")

compare_revisions() ready.


### Running a Revision Comparison

You need **two PDF files** of the same document at different versions/dates.

Good real test case: AUTOSAR publishes multiple dated releases of the same
document (your Q11/completeness output already surfaced release dates like
`2018-10-31` and `2017-12-08` inside this PDF's own change history). Download
an older release of `AUTOSAR_CP_EXP_LayeredSoftwareArchitecture.pdf` from
autosar.org to use as Version A, and keep your current file as Version B.

If you don't have a second real revision handy yet, you can sanity-check the
function using the *same* PDF for both paths first — it should report 0 added,
0 removed, 0 changed (all pages UNCHANGED), which confirms the diff engine
itself is working correctly before you test it on real revisions.

In [5]:
PDF_PATH_A = r"C:\Users\Ananaya Keerthi\Downloads\AUTOSAR_CP_EXP_LayeredSoftwareArchitecture_OLDER.pdf"

PDF_PATH_B = r"C:\Users\Ananaya Keerthi\Downloads\AUTOSAR_CP_EXP_LayeredSoftwareArchitecture.pdf"

# Sanity check first (same file vs itself)
# result = compare_revisions(PDF_PATH_B, PDF_PATH_B)

# Real comparison: OLD version vs CURRENT version
# result = compare_revisions(PDF_PATH_A, PDF_PATH_B)

In [16]:
result = compare_revisions(PDF_PATH_A, PDF_PATH_B)

Extracting and chunking Version A...
Version A: 183 pages, 338 chunks
Extracting and chunking Version B...
Version B: 185 pages, 339 chunks
Computing deterministic page-level diff...
Diff summary: 2 added, 0 removed, 118 changed, 65 unchanged pages

REVISION COMPARISON REPORT

ADDED:
- Finding — Page 184 (Version B)

REMOVED:
- None identified

CHANGED:
- Finding — Page 155: The description of Error Handling, Reporting and Diagnostic has been modified. The new version includes a more detailed explanation of error detection, reporting, and configuration, as well as the interaction between the calling module and the error handling process. Additionally, the version number has been changed from R23-11 to R24-11.

POTENTIAL INCONSISTENCIES:
- Finding — Page 155 (Version A and Version B): The changes between the two versions are mostly descriptive and do not indicate any potential inconsistencies.

OVERALL ASSESSMENT:
The revision comparison report indicates that most of the changes are min

In [13]:
import os
downloads = r"C:\Users\Ananaya Keerthi\Downloads"
for f in os.listdir(downloads):
    if "AUTOSAR" in f.upper():
        print(f)

AUTOSAR_CP_EXP_LayeredSoftwareArchitecture.pdf
AUTOSAR_CP_EXP_LayeredSoftwareArchitecture_OLDER.pdf
Case_Study_1_AUTOSAR_HLD_Assistant (1).ipynb
Case_Study_1_AUTOSAR_HLD_Assistant.ipynb


In [17]:
result = compare_revisions(PDF_PATH_A, PDF_PATH_B, max_changed_pages=30)

Extracting and chunking Version A...
Version A: 183 pages, 338 chunks
Extracting and chunking Version B...
Version B: 185 pages, 339 chunks
Computing deterministic page-level diff...
Diff summary: 2 added, 0 removed, 118 changed, 65 unchanged pages

The comparison of two versions of a document (VERSION A and VERSION B) reveals several changes and similarities. Here's a summary of the analysis:

**Similarities:**

1. Both versions have a similar structure, with similar headings and subheadings.
2. Many sections, such as Architecture, Configuration, and Integration and Runtime Aspects, have similar content in both versions.
3. Some sections, like Error Handling, Reporting and Diagnostic and Energy Management, have similar content in both versions.

**Changes:**

1. Version B has a more detailed table of contents, with more headings and subheadings.
2. In the section on Integration and Runtime Aspects, VERSION B provides more specific details on the E2E wrapper approach, including the log

In [18]:
import json
import os
from datetime import datetime

EXPORT_DIR = "hld_analysis_exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

def export_result(label, content, sources=None, metadata=None):
    """
    Save any analysis output (Q&A answer, report, comparison, etc.) to disk
    as both a readable Markdown file and a structured JSON file.

    label: short name for this output, e.g. "architecture_extraction"
    content: the text answer/report string
    sources: optional list of source chunks/pages used (for traceability)
    metadata: optional dict of extra info (e.g. query used, top_k, timestamp)
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_filename = f"{label}_{timestamp}"

    # --- Markdown export (human-readable) ---
    md_path = os.path.join(EXPORT_DIR, f"{base_filename}.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(f"# {label.replace('_', ' ').title()}\n\n")
        f.write(f"*Generated: {timestamp}*\n\n")
        if metadata:
            f.write("## Metadata\n")
            for k, v in metadata.items():
                f.write(f"- **{k}**: {v}\n")
            f.write("\n")
        f.write("## Output\n\n")
        f.write(content)
        f.write("\n")

    # --- JSON export (machine-readable) ---
    json_path = os.path.join(EXPORT_DIR, f"{base_filename}.json")
    export_data = {
        "label": label,
        "timestamp": timestamp,
        "metadata": metadata or {},
        "content": content,
        "sources": sources or [],
    }
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print(f"Exported: {md_path}")
    print(f"Exported: {json_path}")
    return md_path, json_path


print("export_result() ready.")

export_result() ready.


In [20]:
export_result(
    "revision_comparison",
    result["answer"],
    metadata={
        "pdf_a": PDF_PATH_A,
        "pdf_b": PDF_PATH_B,
        "diff_summary": result["summary_counts"]
    }
)

Exported: hld_analysis_exports\revision_comparison_20260910_100728.md
Exported: hld_analysis_exports\revision_comparison_20260910_100728.json


('hld_analysis_exports\\revision_comparison_20260910_100728.md',
 'hld_analysis_exports\\revision_comparison_20260910_100728.json')

In [28]:
export_result(
    "completeness_check",
    completeness,
    sources=[s["page"] for s in completeness_sources],
    metadata={"query": "Check the AUTOSAR architecture information for missing, incomplete, or potentially inconsistent information."}
)

Exported: hld_analysis_exports\completeness_check_20260910_101322.md
Exported: hld_analysis_exports\completeness_check_20260910_101322.json


('hld_analysis_exports\\completeness_check_20260910_101322.md',
 'hld_analysis_exports\\completeness_check_20260910_101322.json')

In [29]:
export_result(
    "revision_comparison",
    result["answer"],
    metadata={
        "pdf_a": PDF_PATH_A,
        "pdf_b": PDF_PATH_B,
        "diff_summary": result["summary_counts"]
    }
)

Exported: hld_analysis_exports\revision_comparison_20260910_101450.md
Exported: hld_analysis_exports\revision_comparison_20260910_101450.json


('hld_analysis_exports\\revision_comparison_20260910_101450.md',
 'hld_analysis_exports\\revision_comparison_20260910_101450.json')

In [33]:
export_result(
    "architecture_extraction",
    architecture,
    sources=[s["page"] for s in architecture_sources],
    metadata={"query": "extract architecture components/interfaces/layers"}
)

Exported: hld_analysis_exports\architecture_extraction_20260910_101846.md
Exported: hld_analysis_exports\architecture_extraction_20260910_101846.json


('hld_analysis_exports\\architecture_extraction_20260910_101846.md',
 'hld_analysis_exports\\architecture_extraction_20260910_101846.json')

In [37]:
export_result(
    "dependency_mapping",
    dependencies,
    sources=[s["page"] for s in dependency_sources],
    metadata={"query": "extract dependencies"}
)

Exported: hld_analysis_exports\dependency_mapping_20260910_102135.md
Exported: hld_analysis_exports\dependency_mapping_20260910_102135.json


('hld_analysis_exports\\dependency_mapping_20260910_102135.md',
 'hld_analysis_exports\\dependency_mapping_20260910_102135.json')

In [41]:
export_result(
    "component_report_rte",
    rte_report,
    sources=[s["page"] for s in rte_sources],
    metadata={"component": "RTE"}  # adjust based on what you actually queried
)

Exported: hld_analysis_exports\component_report_rte_20260910_102347.md
Exported: hld_analysis_exports\component_report_rte_20260910_102347.json


('hld_analysis_exports\\component_report_rte_20260910_102347.md',
 'hld_analysis_exports\\component_report_rte_20260910_102347.json')

In [42]:
import os
for f in sorted(os.listdir("hld_analysis_exports")):
    print(f)

architecture_extraction_20260910_101846.json
architecture_extraction_20260910_101846.md
completeness_check_20260910_101322.json
completeness_check_20260910_101322.md
component_report_rte_20260910_102347.json
component_report_rte_20260910_102347.md
dependency_mapping_20260910_102135.json
dependency_mapping_20260910_102135.md
revision_comparison_20260910_100723.json
revision_comparison_20260910_100723.md
revision_comparison_20260910_100728.json
revision_comparison_20260910_100728.md
revision_comparison_20260910_101450.json
revision_comparison_20260910_101450.md


In [43]:
import os
print(os.path.abspath("hld_analysis_exports"))

C:\Users\Ananaya Keerthi\Downloads\hld_analysis_exports


In [44]:
import os
import glob
from datetime import datetime

def combine_exports(export_dir="hld_analysis_exports", output_name="COMBINED_REPORT"):
    """
    Combines all exported .md files into a single master report,
    ordered logically by analysis type rather than by timestamp.
    """
    # Preferred order for readability
    order = [
        "architecture_extraction",
        "dependency_mapping",
        "component_report_rte",
        "completeness_check",
        "revision_comparison",
    ]

    all_files = glob.glob(os.path.join(export_dir, "*.md"))

    # For each label, keep only the LATEST file (highest timestamp)
    latest_by_label = {}
    for f in all_files:
        filename = os.path.basename(f)
        for label in order:
            if filename.startswith(label):
                if label not in latest_by_label or filename > os.path.basename(latest_by_label[label]):
                    latest_by_label[label] = f

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_path = os.path.join(export_dir, f"{output_name}_{timestamp}.md")

    with open(combined_path, "w", encoding="utf-8") as out:
        out.write("# AUTOSAR HLD Document Analysis — Combined Report\n\n")
        out.write(f"*Generated: {timestamp}*\n\n")
        out.write("---\n\n")
        out.write("## Table of Contents\n")
        for label in order:
            if label in latest_by_label:
                out.write(f"- {label.replace('_', ' ').title()}\n")
        out.write("\n---\n\n")

        for label in order:
            if label in latest_by_label:
                with open(latest_by_label[label], "r", encoding="utf-8") as f:
                    out.write(f.read())
                out.write("\n\n---\n\n")

    print(f"Combined report saved: {combined_path}")
    return combined_path

combine_exports()

Combined report saved: hld_analysis_exports\COMBINED_REPORT_20260910_102859.md


'hld_analysis_exports\\COMBINED_REPORT_20260910_102859.md'